In [2]:
import os
import camelot
import pandas as pd
from tqdm import tqdm
from IPython.display import display
import warnings
import re
import numpy as np
import pdfplumber

warnings.filterwarnings("ignore")

folder = os.getcwd()
pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


def extract_numbers(name):

    nums = re.findall(r'\d+', name)
    return tuple(map(int, nums))

replacements = {
    'Ⅰ': 'I',
    'Ⅱ': 'II',
    'Ⅲ': 'III',
    'Ⅳ': 'IV',
    'Ⅴ': 'V',
    'Ⅵ': 'VI',
    'Ⅶ': 'VII',
    'Ⅷ': 'VIII',
    'Ⅸ': 'IX',
    'Ⅹ': 'X'
}

def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):

    os.makedirs(out_dir, exist_ok=True)

    tables_lattice = camelot.read_pdf(
        pdf_path,
        pages=pages,
        flavor="lattice",
        line_scale=80
    )

    lattice_pages = set()
    text_pages = set()

    with pdfplumber.open(pdf_path) as pdf:

        for i, t in enumerate(tables_lattice):

            page_num = int(t.page)          
            page_idx = page_num - 1        
            lattice_pages.add(page_num)

            t.to_csv(
                os.path.join(out_dir, f"lattice_p{page_num}_{i}.csv"),
                index=False
            )

            page = pdf.pages[page_idx]

            x1, y1, x2, y2 = t._bbox
            page_w = page.width
            page_h = page.height

            table_top = page_h - y2

            left   = max(0, min(x1, page_w))
            right  = max(0, min(x2, page_w))
            top    = max(0, min(table_top - title_height, page_h))
            bottom = max(0, min(table_top, page_h))

            if right > left and bottom > top:
                title_box = (left, top, right, bottom)
                title_text = page.crop(title_box, strict=False).extract_text(
                    x_tolerance=2,
                    y_tolerance=2
                ) or ""

                if title_text.strip():
                    text_pages.add(page_num)
            else:
                title_text = ""

            with open(
                os.path.join(out_dir, f"text_p{page_num}_{i}.txt"),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(title_text.strip())

    return None       


def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None

def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'Qty' in df.columns:
        agg_map = {
            "Qty": "sum",
        }
    elif 'Код материалов' in df.columns:
        agg_map = {
            "Кол-во": "sum",
        }
    elif '件数' in df.columns:
        agg_map = {
            '件数': 'sum'
        }
    elif 'Units' in df.columns:
        agg_map = {
            'Units': 'sum'
        }

    if "Item and Spec" in df.columns:

        agg_map["Item and Spec"] = join_ordered
    if "Remark" in df.columns:
        agg_map["Remark"] = join_ordered

    if 'Наименование и спецификация' in df.columns:
        agg_map['Наименование и спецификация'] = join_ordered

    if 'Примечание' in df.columns:
        agg_map['Примечание'] = join_ordered

    if 'Версия' in df.columns:
        agg_map['Версия'] = join_ordered

    if 'Rev.' in df.columns:
        agg_map['Rev.'] = join_ordered

    if "名称及规格" in df.columns:
        agg_map["名称及规格"] = join_ordered

    if "备注" in df.columns:
        agg_map["备注"] = join_ordered

    if "Section" in df.columns:
        agg_map["Section"] = join_ordered

    if 'Material No.' in df.columns:
        df_agg = df.groupby("Material No.", as_index=False).agg(agg_map)
    elif 'Код материалов' in df.columns:
        df_agg = df.groupby("Код материалов", as_index=False).agg(agg_map)
    elif '物料编码' in df.columns:
        df_agg = df.groupby("物料编码", as_index=False).agg(agg_map)


    return df_agg

In [36]:
b = 14
e = b+1

for i, pth in enumerate(tqdm(pdf_files[b:e])):
    
    filename = os.path.basename(pth)          
    filename = os.path.splitext(filename)[0]
    print(f'Обработка файла: {filename}')

    # extract_tables_camelot(pth, filename, pages="all", title_height=800)

    dfs = []

    for file in sorted(os.listdir(filename), key=extract_numbers):

        if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
            continue

        path = os.path.join(filename, file)

        if os.path.getsize(path) == 0:  
            continue
        df_i = pd.read_csv(path, converters={1: str})

        if (
            df_i.empty
            or df_i.shape[1] < 4
            or not any(col in df_i.columns for col in ['NO.', 'Код материалов', 'No.', 'Код мат-ла', 'Serial No.', 'Part No.', '物料编码', 'п/п  Код материалов', 'Код', 'Material No.', 'Material  Code'])
            ):
            continue

        # если китаец насрал пробел в начале таблицы
        if 'NO.' in df_i.iloc[0].values or 'Material No.' in df_i.iloc[0].values:
            df_i.columns = df_i.iloc[0]  
            df_i = df_i[1:].reset_index(drop=True) 

        for col in ['NO.', 'No.', '№ п/п', '№ \nп/п', 'Serial No.', '序号', '№ \n\nп/п', '№ \r\nп/п', '№', 'N\r\no.', 'No\r\n.', 'Serial  number']:
            if col in df_i.columns:
                df_i = df_i.drop(col, axis=1)

        idx = file.replace("lattice_", "").rsplit(".", 1)[0]
        section_val = os.path.join(filename, f"text_{idx}.txt")
        
        with open(section_val, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        skip_titles = [
            'Руководство по деталъная структура',
            'Альбом чертежей деталей и узлов коленчатого подъемника для высотных работ',
            'Альбом чертежей деталей и компонентов',
            'Parts Manual',
            '零部件图册',
            'Translated by Google',
            'anslated by Google'
        ]

        if lines and lines[0] in skip_titles:
            lines = lines[1:]
                
        if lines and lines[0] in ['Альбом чертежей деталей и узлов коленчатого', 'Альбом чертежей деталей и узлов коленчатого подъемника для']:
            lines = lines[2:]

        lines_joined = " ".join(lines)
            
        df_i["Section"] = lines_joined
        
        df_i = df_i[df_i.iloc[:, 0].replace('', pd.NA).notna()]
        display(df_i)
        df_i = df_i[df_i.iloc[:, 0] != '/']
        dfs.append(df_i)

    df_all = pd.concat(dfs, ignore_index=True)
    display(df_all)
    df_all.to_excel('dfs.xlsx', index=False)

    for old_col in ['во', 'Кол-\nво', 'Кол-\r\nво', 'Кол-в\nо \nштук', 'Кол-\nво \nшту\nк', 'Кол-во \nштук', 'Cantidad', 'Кол-во\n\nштук', 'Кол-во\r\nштук', 'Кол-\n\nво \n\nшту\n\nк', 'Кол-во \r\nштук', 'Кол-в\r\nо \r\nштук', 'Кол-\r\nво \r\nшту\r\nк', 'Кол-во штук', 'Кол-в', 'Колич\r\nество', 'Кол-в\r\nо\r\nштук', 'Колич ество', 'Кол-во Примечание', 'Number  of  pieces']:
        if old_col in df_all.columns.tolist():
            if 'Кол-во' in df_all.columns.tolist():
                df_all['Кол-во'] = df_all.loc[:, old_col].combine_first(df_all['Кол-во'])
            else:
                df_all['Кол-во'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Кол-во' in df_all.columns:
        col = df_all.pop("Кол-во")  
        df_all.insert(2, "Кол-во", col) 

    for old_col in ['Наименование и \nспецификация', 'Наименование и \r\nспецификация', 'Наименование и\n\nспецификация', 'Nombre y especificación', 'Наименование и \r\nспецификация', 'Наименование и\r\nспецификация', 'Наименование и', 'Наименование\r\nи', 'Наименование']:
        if old_col in df_all.columns.tolist():
            if 'Наименование и спецификация' in df_all.columns:
                df_all['Наименование и спецификация'] = df_all.loc[:, old_col].combine_first(df_all['Наименование и спецификация'])
            else:
                df_all['Наименование и спецификация'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Наименование и спецификация' in df_all.columns:
        col = df_all.pop("Наименование и спецификация")  
        df_all.insert(1, "Наименование и спецификация", col) 

    for old_col in ['Name and Spec.', 'Spec and Item', 'Name  and  specifications']:
        if old_col in df_all.columns:
            if 'Item and Spec' in df_all.columns:
                df_all['Item and Spec'] = df_all.loc[:, old_col].combine_first(df_all['Item and Spec'])
            else:
                df_all['Item and Spec'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Item and Spec' in df_all.columns:
        col = df_all.pop('Item and Spec')  
        df_all.insert(1, 'Item and Spec', col) 

    for old_col in ['Примечани\nе', 'Observaciones', 'Примечани\n\nе', 'Примечани\r\nе', '№ позиции', 'Примеч\r\nание']:
        if old_col in df_all.columns:
            if 'Примечание' in df_all.columns:
                df_all['Примечание'] = df_all.loc[:, old_col].combine_first(df_all['Примечание'])
            else:
                df_all['Примечание'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Примечание' in df_all.columns:
        col = df_all.pop('Примечание')  
        df_all.insert(3, 'Примечание', col) 

    for old_col in ['Qty.', 'Unit', 'Qt\r\ny']:
        if old_col in df_all.columns:
            if 'Qty' in df_all.columns:
                df_all['Qty'] = df_all.loc[:, old_col].combine_first(df_all['Qty'])
            else:
                df_all['Qty'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Qty' in df_all.columns:
        col = df_all.pop('Qty')  
        df_all.insert(2, 'Qty', col) 
        
    for old_col in ['Código de material', 'Код мат-ла', 'п/п  Код материалов', 'Код']:
        if old_col in df_all.columns:
            if 'Код материалов' in df_all.columns:
                df_all['Код материалов'] = df_all.loc[:, old_col].combine_first(df_all['Код материалов'])
            else:
                df_all['Код материалов'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Код материалов' in df_all.columns:
        col = df_all.pop('Код материалов')  
        df_all.insert(0, 'Код материалов', col) 

    for old_col in ['"Версия"', 'Версия  Кол-']:
        if old_col in df_all.columns:
            if 'Версия' in df_all.columns:
                df_all['Версия'] = df_all.loc[:, old_col].combine_first(df_all['Версия'])
            else:
                df_all['Версия'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)

    for old_col in ['Part No.', 'Material Code', 'Material  Code']:
        if old_col in df_all.columns:
            if 'Material No.' in df_all.columns:
                df_all['Material No.'] = df_all.loc[:, old_col].combine_first(df_all['Material No.'])
            else:
                df_all['Material No.'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Material No.' in df_all.columns:
        col = df_all.pop('Material No.')  
        df_all.insert(0, 'Material No.', col) 

    for old_col in ['Note']:
        if old_col in df_all.columns:
            if 'Remark' in df_all.columns:
                df_all['Remark'] = df_all.loc[:, old_col].combine_first(df_all['Remark'])
            else:
                df_all['Remark'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Remark' in df_all.columns:
        col = df_all.pop('Remark')  
        df_all.insert(3, 'Remark', col) 

    for col in ['Кол-во', 'Qty', '件数']:
        if col in df_all.columns:
            df_all[col] = df_all[col].replace({'/': 0, '、': 0, 'A': 0, '-': 0})
            df_all[col] = df_all[col].fillna(0)
            df_all[col] = df_all[col].astype(str).str.replace(',', '.', regex=False)
            df_all[col] = pd.to_numeric(df_all[col], errors='coerce').fillna(0)
            df_all[col] = df_all[col].apply(lambda x: int(x) if x.is_integer() else x)
    
    if 'Код материалов' in df_all.columns:
        df_all['Код материалов'] = df_all['Код материалов'].astype(str).str.replace('\xa0', ' ', regex=False).str.strip()
        extracted = df_all['Код материалов'].str.extract(r'^(\S+)\s+(.*)$')
        mask = extracted[0].notna()
        df_all.loc[mask, 'Наименование и спецификация'] = extracted.loc[mask, 1]
        df_all.loc[mask, 'Код материалов'] = extracted.loc[mask, 0]

    df_all.to_excel('df_all.xlsx', index=False)
    df_all = pd.read_excel('df_all.xlsx')
    test_1 = pd.read_excel('dfs.xlsx')
    assert test_1.shape[0] == df_all.shape[0]

    for col in ['Material No.', 'Код материалов', '物料编码']:
        if col in df_all.columns:
            df_all = df_all[df_all[col] != '']
            df_all = df_all[~df_all[col].astype(str).str.startswith('/')]
            df_all = df_all[~df_all[col].isna()]

    display(df_all.head())

    if 'Material No.' in df_all.columns and df_all['Material No.'].astype(str).str.contains('NO-NUMBER').any():
        df_all['Material No.'] = df_all['Material No.'].str.split('\n').str[0]
        df_all = df_all[~df_all['Material No.'].astype(str).str.startswith('NO-NUMBER')]

    if 'Код материалов' in df_all.columns and df_all['Код материалов'].astype(str).str.contains('NO-NUMBER').any():
        df_all = df_all[~df_all['Код материалов'].astype(str).str.startswith('NO-NUMBER')]

    cols = df_all.select_dtypes(include='object').columns
    df_all[cols] = (df_all[cols].replace(r'\s*\n\s*', ' ', regex=True))
    df_all[cols] = df_all[cols].replace(replacements, regex=True)

    if 'Item and Spec' in df_all.columns:
        df_all['Item and Spec'] = df_all['Item and Spec'].replace('/', np.nan)
    
    if '物料编码' in df_all.columns:
        mask = df_all['物料编码'].str.contains(r'^\d+\s+\D', na=False)
        extracted = df_all.loc[mask, '物料编码'].str.extract(r'^(\d+)\s+(.*)$')
        df_all.loc[mask, '物料编码'] = extracted[0]
        df_all.loc[mask, '名称及规格'] = extracted[1]
        
    df_merged = merge_parts(df_all) 
   
    assert (df_all.shape[0] - df_all.iloc[:, 0].duplicated().sum() - df_merged.shape[0]) == 0

    lst = ['ZS1623RT', 'ZA10RJE', 'ZA14J', 'ZA14JE-Li', 'ZA14NJE', 'ZA16J', 'ZA16JERT', 'ZA18J', 'ZA20J', 'ZA20JE', 'ZA20JERT', 'ZA22J-V' ,'ZA22J', 'ZA24J', 'ZA32J(H-образ.)', 'ZA32J(X-образ.)', 'ZT42J-V', 'ZA42J', '', '', '', '', '', '', '', '', '' ,'', '', 'ZT20J', 'ZT22J-V', 'ZT26J', 'ZT26JS-V', 'ZT30J', 'ZT32J-V', 'ZT32J', 'ZT34J', 'ZT38J', 'ZT42J', 'ZT51J', 'ZT58J', 'ZT72J-V', 'ZT82J', 'ZTH3507', 'ZTH3513', 'ZTH3513', 'ZX23AE']
    df_merged['Models'] = lst[b]

    df_merged.to_excel(f'{filename}.xlsx', index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: ZA32J ( H -образ.)


,Код материалов,Наименование и \r\nспецификация,Кол-\r\nво,Примечание,Section
0,1040200503,Гайка,2,JB/T889.1,Рабочая платформа 00774000100000000
1,1040302480,Шайба 8-200HV,4,JB/T96.1,Рабочая платформа 00774000100000000
2,00773400100261050,Подшипник,2,NaN,Рабочая платформа 00774000100000000
3,00773400100261060,Болт φ22,2,NaN,Рабочая платформа 00774000100000000
4,00773400100260020,Защитный кожух,1,NaN,Рабочая платформа 00774000100000000
5,1040005844,Болт M6×10-A2-70,4,JB/T5783,Рабочая платформа 00774000100000000
6,1040300771,Шайба 6-200HV,8,JB/T97.1,Рабочая платформа 00774000100000000
7,00773400100261030,Пластина II,1,NaN,Рабочая платформа 00774000100000000
8,00773400100261080,Упорная планка,2,NaN,Рабочая платформа 00774000100000000
9,00773406200210000,Электрический шкаф \r\nуправления платформой в...,1,NaN,Рабочая платформа 00774000100000000


,Код материалов,Наименование и спецификация,Кол-во,Примечание,Section
0,1159901036,Пружинная петля,2,NaN,Рабочая платформа 00774000100000000
1,1040300514,Шайба 5-200HV,12,JB/T97.1,Рабочая платформа 00774000100000000
2,1029904911,Пружинящая шайба,12,GB93-304,Рабочая платформа 00774000100000000
3,1040004672,"Болт M5×10-8,8",12,JB/T5783,Рабочая платформа 00774000100000000
4,1040001640,Болт M6×25-A2-70,2,JB/T5783,Рабочая платформа 00774000100000000
5,1040300771,Шайба 6-200HV,2,JB/T97.1,Рабочая платформа 00774000100000000
6,00774000101000000,Сварной элемент кронштейна \r\nфильтра,1,NaN,Рабочая платформа 00774000100000000
7,1040201896,Гайка M6-8,2,JB/T889.1,Рабочая платформа 00774000100000000
8,00774000100810001,Переходный держатель,1,NaN,Рабочая платформа 00774000100000000
9,1040004619,"Болт M8×35-8,8",4,JB/T5783,Рабочая платформа 00774000100000000


,Код материалов,Наименование и \r\nспецификация,Кол-во,Примечание,Section
0,00774100200400000,Верхний шатун,1,NaN,Консоль 00774100200000000
1,00774000200001300,Плита трубопровода,2,NaN,Консоль 00774100200000000
2,00773700200001100,Зажимная планка \r\nтрубопровода,2,NaN,Консоль 00774100200000000
3,1040103262,Винт M6×45-8.8,8,NaN,Консоль 00774100200000000
4,00774100200001010,Щиток трубы,1,NaN,Консоль 00774100200000000
5,1040004522,"Болт M8×50-8,8",4,JB/T5783,Консоль 00774100200000000
6,00773400200001010,Подшипник,12,NaN,Консоль 00774100200000000
7,1040301515,Шайба 8-200HV,10,JB/T97.1,Консоль 00774100200000000
8,1040004590,"Болт M8×14-8,8",4,JB/T5783,Консоль 00774100200000000
9,00771400211200000,Подвижный трубный хомут,2,NaN,Консоль 00774100200000000


...

,Код материалов,Наименование и спецификация,Кол-во,Примечание,Section
0,00774006200210000,Электрический шкаф управления \r\nплатформой,1,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
1,00773406200210000,Электрический шкаф управления \r\nплатформой,0,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
2,1020521022,Переключатель защиты от \r\nсдавливания,1,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
3,/,Клапан подъема маховой стрелы,/,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
4,/,Клапан левого качания платформы,/,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
5,/,Клапан опускания при \r\nвыравнивании,/,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
6,/,Клапан опускания телескопической \r\nстрелы,/,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
7,/,Клапан правого качания платформы,/,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
8,/,Клапан повышения при \r\nвыравнивании,/,NaN,Пучок кабеля платформы 00774106200230040 Модиф...
9,1021404016,Педальный переключатель \r\nплатформы,/,NaN,Пучок кабеля платформы 00774106200230040 Модиф...


,Код материалов,Наименование и спецификация,Кол-во,Примечание,Section
0,1020903397,Аккумулятор,1,NaN,Питательный провод 00774106200230050
1,1020605356,Кольцевой предохранитель,2,NaN,Питательный провод 00774106200230050
2,0067220632181001A-c,Опора кольцевого предохранителя,2,NaN,Питательный провод 00774106200230050
3,1020402443,Изоляционная гайка,2,NaN,Питательный провод 00774106200230050
4,00774106200234040,Соединительная линия между \r\nвспомогательным...,1,NaN,Питательный провод 00774106200230050
5,00774106200234050,Провод от положительного полюса \r\nаккумулято...,1,NaN,Питательный провод 00774106200230050
6,00774006200234020,Соединительный провод от \r\nотрицательного по...,1,NaN,Питательный провод 00774106200230050
7,1029900667,Ручной переключатель,1,NaN,Питательный провод 00774106200230050
8,00774106200234030,Соединительный провод от ручного \r\nпереключа...,1,NaN,Питательный провод 00774106200230050
9,00774106200234070,Соединительный провод между \r\nдвигателем и м...,1,NaN,Питательный провод 00774106200230050


,Код материалов,Наименование и \r\nспецификация,Кол-\r\nво,Примечание,Section,Наименование и спецификация,Кол-во,Примечани\r\nе,Количество,Кол-во Примечание,Unnamed: 4,Unnamed: 3,Кол-во Примечание,№\r\n п/п
0,1040200503,Гайка,2.0,JB/T889.1,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1040302480,Шайба 8-200HV,4.0,JB/T96.1,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00773400100261050,Подшипник,2.0,NaN,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00773400100261060,Болт φ22,2.0,NaN,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00773400100260020,Защитный кожух,1.0,NaN,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,00774106200234070,NaN,NaN,NaN,Питательный провод 00774106200230050,Соединительный провод между \r\nдвигателем и м...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
737,00774006200234090,NaN,NaN,NaN,Питательный провод 00774106200230050,Соединительный провод между реле \r\nи подогре...,/,NaN,NaN,NaN,NaN,NaN,NaN,NaN
738,1020005456,NaN,NaN,NaN,Питательный провод 00774106200230050,Контактор постоянного тока,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
739,00774006200234060,NaN,NaN,NaN,Питательный провод 00774106200230050,Пусковой мотор соединяется с \r\nгенератором,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Код материалов,Наименование и спецификация,Кол-во,Примечание,Section,Количество,Кол-во Примечание,Unnamed: 4,Unnamed: 3,№\n\n п/п
0,1040200503,Гайка,2,JB/T889.1,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN
1,1040302480,Шайба 8-200HV,4,JB/T96.1,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN
2,00773400100261050,Подшипник,2,NaN,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN
3,00773400100261060,Болт φ22,2,NaN,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN
4,00773400100260020,Защитный кожух,1,NaN,Рабочая платформа 00774000100000000,NaN,NaN,NaN,NaN,NaN


100%|██████████| 1/1 [00:01<00:00,  1.88s/it]


- Спросить у Миши про запчасти, в описании которых встречается ещё одна модель, помимо основной, в которой они встречаются
- Что делать с такими в описании (Cubota): ZT138J-ZT24JE-V Turntable Decal Assembly - что значит тире?

In [38]:
df_all = pd.read_excel('dfs_2.xlsx')
display(df_all.info())
for old_col in ['во', 'Кол-\nво', 'Кол-\r\nво', 'Кол-в\nо \nштук', 'Кол-\nво \nшту\nк', 'Кол-во \nштук', 'Cantidad', 'Кол-во\n\nштук', 'Кол-во\r\nштук', 'Кол-\n\nво \n\nшту\n\nк', 'Кол-во \r\nштук', 'Кол-в\r\nо \r\nштук', 'Кол-\r\nво \r\nшту\r\nк', 'Кол-во штук', 'Кол-в', 'Колич\r\nество', 'Кол-в\r\nо\r\nштук', 'Колич ество', 'Кол-во Примечание', 'Number  of  pieces']:
    if old_col in df_all.columns.tolist():
        if 'Кол-во' in df_all.columns.tolist():
            df_all['Кол-во'] = df_all.loc[:, old_col].combine_first(df_all['Кол-во'])
        else:
            df_all['Кол-во'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Кол-во' in df_all.columns:
    col = df_all.pop("Кол-во")  
    df_all.insert(2, "Кол-во", col) 

for old_col in ['Наименование и \nспецификация', 'Наименование и \r\nспецификация', 'Наименование и\n\nспецификация', 'Nombre y especificación', 'Наименование и \r\nспецификация', 'Наименование и\r\nспецификация', 'Наименование и', 'Наименование\r\nи', 'Наименование']:
    if old_col in df_all.columns.tolist():
        if 'Наименование и спецификация' in df_all.columns:
            df_all['Наименование и спецификация'] = df_all.loc[:, old_col].combine_first(df_all['Наименование и спецификация'])
        else:
            df_all['Наименование и спецификация'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Наименование и спецификация' in df_all.columns:
    col = df_all.pop("Наименование и спецификация")  
    df_all.insert(1, "Наименование и спецификация", col) 

for old_col in ['Name and Spec.', 'Spec and Item', 'Name  and  specifications']:
    if old_col in df_all.columns:
        if 'Item and Spec' in df_all.columns:
            df_all['Item and Spec'] = df_all.loc[:, old_col].combine_first(df_all['Item and Spec'])
        else:
            df_all['Item and Spec'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Item and Spec' in df_all.columns:
    col = df_all.pop('Item and Spec')  
    df_all.insert(1, 'Item and Spec', col) 

for old_col in ['Примечани\nе', 'Observaciones', 'Примечани\n\nе', 'Примечани\r\nе', '№ позиции', 'Примеч\r\nание']:
    if old_col in df_all.columns:
        if 'Примечание' in df_all.columns:
            df_all['Примечание'] = df_all.loc[:, old_col].combine_first(df_all['Примечание'])
        else:
            df_all['Примечание'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Примечание' in df_all.columns:
    col = df_all.pop('Примечание')  
    df_all.insert(3, 'Примечание', col) 

for old_col in ['Qty.', 'Unit', 'Qt\r\ny']:
    if old_col in df_all.columns:
        if 'Qty' in df_all.columns:
            df_all['Qty'] = df_all.loc[:, old_col].combine_first(df_all['Qty'])
        else:
            df_all['Qty'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Qty' in df_all.columns:
    col = df_all.pop('Qty')  
    df_all.insert(2, 'Qty', col) 
    
for old_col in ['Código de material', 'Код мат-ла', 'п/п  Код материалов', 'Код']:
    if old_col in df_all.columns:
        if 'Код материалов' in df_all.columns:
            df_all['Код материалов'] = df_all.loc[:, old_col].combine_first(df_all['Код материалов'])
        else:
            df_all['Код материалов'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Код материалов' in df_all.columns:
    col = df_all.pop('Код материалов')  
    df_all.insert(0, 'Код материалов', col) 

for old_col in ['"Версия"', 'Версия  Кол-']:
    if old_col in df_all.columns:
        if 'Версия' in df_all.columns:
            df_all['Версия'] = df_all.loc[:, old_col].combine_first(df_all['Версия'])
        else:
            df_all['Версия'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)

for old_col in ['Part No.', 'Material Code', 'Material  Code']:
    if old_col in df_all.columns:
        if 'Material No.' in df_all.columns:
            df_all['Material No.'] = df_all.loc[:, old_col].combine_first(df_all['Material No.'])
        else:
            df_all['Material No.'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Material No.' in df_all.columns:
    col = df_all.pop('Material No.')  
    df_all.insert(0, 'Material No.', col) 

for old_col in ['Note']:
    if old_col in df_all.columns:
        if 'Remark' in df_all.columns:
            df_all['Remark'] = df_all.loc[:, old_col].combine_first(df_all['Remark'])
        else:
            df_all['Remark'] = df_all[old_col]
        df_all = df_all.drop(columns=old_col)
if 'Remark' in df_all.columns:
    col = df_all.pop('Remark')  
    df_all.insert(3, 'Remark', col) 

for col in ['Кол-во', 'Qty', '件数']:
    if col in df_all.columns:
        df_all[col] = df_all[col].replace({'/': 0, '、': 0, 'A': 0, '-': 0})
        df_all[col] = df_all[col].fillna(0)
        df_all[col] = df_all[col].astype(str).str.replace(',', '.', regex=False)
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce').fillna(0)
        df_all[col] = df_all[col].apply(lambda x: int(x) if x.is_integer() else x)

if 'Код материалов' in df_all.columns:
    df_all['Код материалов'] = df_all['Код материалов'].astype(str).str.replace('\xa0', ' ', regex=False).str.strip()
    extracted = df_all['Код материалов'].str.extract(r'^(\S+)\s+(.*)$')
    mask = extracted[0].notna()
    df_all.loc[mask, 'Наименование и спецификация'] = extracted.loc[mask, 1]
    df_all.loc[mask, 'Код материалов'] = extracted.loc[mask, 0]

df_all.to_excel('df_all.xlsx', index=False)
df_all = pd.read_excel('df_all.xlsx')
test_1 = pd.read_excel('dfs.xlsx')
assert test_1.shape[0] == df_all.shape[0]

for col in ['Material No.', 'Код материалов', '物料编码']:
    if col in df_all.columns:
        df_all = df_all[df_all[col] != '']
        df_all = df_all[~df_all[col].astype(str).str.startswith('/')]
        df_all = df_all[~df_all[col].isna()]

display(df_all.head())

if 'Material No.' in df_all.columns and df_all['Material No.'].astype(str).str.contains('NO-NUMBER').any():
    df_all['Material No.'] = df_all['Material No.'].str.split('\n').str[0]
    df_all = df_all[~df_all['Material No.'].astype(str).str.startswith('NO-NUMBER')]

if 'Код материалов' in df_all.columns and df_all['Код материалов'].astype(str).str.contains('NO-NUMBER').any():
    df_all = df_all[~df_all['Код материалов'].astype(str).str.startswith('NO-NUMBER')]

cols = df_all.select_dtypes(include='object').columns
df_all[cols] = (df_all[cols].replace(r'\s*\n\s*', ' ', regex=True))
df_all[cols] = df_all[cols].replace(replacements, regex=True)

if 'Item and Spec' in df_all.columns:
    df_all['Item and Spec'] = df_all['Item and Spec'].replace('/', np.nan)

if '物料编码' in df_all.columns:
    mask = df_all['物料编码'].str.contains(r'^\d+\s+\D', na=False)
    extracted = df_all.loc[mask, '物料编码'].str.extract(r'^(\d+)\s+(.*)$')
    df_all.loc[mask, '物料编码'] = extracted[0]
    df_all.loc[mask, '名称及规格'] = extracted[1]
    
df_merged = merge_parts(df_all) 

assert (df_all.shape[0] - df_all.iloc[:, 0].duplicated().sum() - df_merged.shape[0]) == 0

lst = ['ZS1623RT', 'ZA10RJE', 'ZA14J', 'ZA14JE-Li', 'ZA14NJE', 'ZA16J', 'ZA16JERT', 'ZA18J', 'ZA20J', 'ZA20JE', 'ZA20JERT', 'ZA22J-V' ,'ZA22J', 'ZA24J', 'ZA32J(H-образ.)', 'ZA32J(X-образ.)', 'ZT42J-V', 'ZA42J', '', '', '', '', '', '', '', '', '' ,'', '', 'ZT20J', 'ZT22J-V', 'ZT26J', 'ZT26JS-V', 'ZT30J', 'ZT32J-V', 'ZT32J', 'ZT34J', 'ZT38J', 'ZT42J', 'ZT51J', 'ZT58J', 'ZT72J-V', 'ZT82J', 'ZTH3507', 'ZTH3513', 'ZTH3513', 'ZX23AE']
df_merged['Models'] = lst[b]

df_merged.to_excel(f'{filename}.xlsx', index=False)

<class 'pandas.DataFrame'>
RangeIndex: 741 entries, 0 to 740
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Код материалов                 741 non-null    object 
 1   Наименование и 

спецификация  697 non-null    str    
 2   Кол-

во                       736 non-null    object 
 3   Примечание                     264 non-null    str    
 4   Section                        741 non-null    str    
 5   Наименование и спецификация    0 non-null      float64
 6   Кол-во                         0 non-null      float64
 7   Примечани

е                   0 non-null      float64
 8   Количество                     0 non-null      float64
 9   Кол-во  Примечание             0 non-null      float64
 10  Unnamed: 4                     0 non-null      float64
 11  Unnamed: 3                     0 non-null      float64
 12  Кол-во Примечание              0 non-null      float64
 13  №

None

TypeError: Invalid value '<StringArray>
[              'Прижимная планка трубопровода I',
                              'Трубный хомут II',
                               'Трубный хомут I',
                                  'Прокладка IV',
                                     'Подшипник',
                              'Монтажная планка',
                              'Распорная втулка',
                      'Стопорный штифт φ20×120*',
                                     'Штырь III',
                                        'Ползун',
                      'Регулирующая прокладка Ⅳ',
                      'Пылезащитная    щетка II',
                                     'Прокладка',
                      'Пылезащитная    щетка IV',
                        'Регулировочная шайба Ⅴ',
                                  'Узлы ползуна',
             'Длинная регулирующая прокладка II',
              'Длинная регулирующая прокладка I',
                                'Длинный сухарь',
 'Сварной 

In [ ]:
df.